In [ ]:
import numpy as np
from macti.PyNoxtli.geo.line import Line
from macti.PyNoxtli.fvm.sDiffusion import sDiffusion1D
from macti.PyNoxtli.fvm.pde import PDE
from macti.PyNoxtli.utils.displayInfo import printInfo
import macti.PyNoxtli.vis.flowix as flx

In [1]:
from macti.PyNoxtli.fvm.tDiffusion import tDiffusion1D

In [ ]:
#
# Propiedades físicas y geométricas de la barra
#
longitud = 0.5 # [m]
TA = 100       # [°C] Temperatura en el extremo izquierdo
TB = 500       # [°C] Temperatura en el extremo derecho
k  = 1000      # [W/m.K] Conductividad térmica
N  = 6         # Número de nodos

In [ ]:
#
# Definición del dominio y condiciones de frontera
#
barra = Line(longitud)
barra.boundaryConditions(dirichlet = {'LEFT':TA, 'RIGHT':TB})
#
# Creación de la malla 
#
malla     = barra.constructMesh(N) # Se construye la malla del dominio
ivx, _, _ = malla.bounds(bi = 1, ei = N-1) # Grados de libertad
nx        = malla.nx    # Número de nodos
nvx       = malla.vx    # Número de volúmenes (celdas)
delta     = malla.dx    # Tamaño de los volúmenes
#
# Arreglo para almacenar la solución
#
T = np.zeros(nvx+2) # Arreglo inicializado con ceros
T[0]  = TA          # Condición de frontera izquierda
T[-1] = TB          # Condición de frontera derecha

print("Temperatura inicial: {}".format(T))

In [ ]:
#
# Impresión de los datos del problema
#
printInfo(Longitud = longitud,
          Temperatura_A = TA,
          Temperatura_B = TB,
          Conductividad = k,
          Nodos = nx, 
          Volúmenes = nvx,
          Delta = delta)

In [ ]:
#
# Definición de la fuente 
#
Su = np.zeros(ivx) # Por ahora no hay fuente
#
# Definición del esquema de disccretización
#
dif_scheme = sDiffusion1D(malla, Su, k)
#
# Definición de la ecuación a resolver
#
laplace = PDE(barra, T)
#
# Creación del sistema lineal y su solución
#
laplace.setNumericalScheme(dif_scheme)
laplace.solve()
print('Solución : {}'.format(T))

In [ ]:
#
# Coordenadas de la malla para FVM
x, _, _ = malla.coordinatesMeshFVM() 
#
# Solución analítica
Ta = 800 * x + 100
#
# Visualización 
axis_par = [{'title':'Malla'},
            {'title':'Solución Numérica', 'xlabel':'x [m]', 'ylabel':'T [$^o$C]'},
            {'title':'Solución Exacta', 'xlabel':'x [m]', 'ylabel':'T [$^o$C]'}]

v = flx.Plotter(3,1,axis_par) # Son 3 renglones y una columna de ejes (Axes).
v.plot(2,x,T, {'marker':'s', 'ls':'-', 'c':'C0', 'label':'Numérica'})
v.plot(3,x,Ta, {'marker':'o', 'ls':'-', 'c':'C1', 'label':'Exacta'})
v.plot_mesh(1, malla, label=True)
v.legend()
v.show()